# Création de la base d'apprentissage

Ce notebook constitue la **réconciliation de données multi-sources** pour construire notre dataset d'entraînement. L'objectif est d'aligner les statistiques de performance des joueurs avec leurs valeurs marchandes respectives, en ayant donc une ligne par joueur par saison.

Nous centralisons alors dans un premier temps des fichiers issus de l'API Soccerdata, regroupant ainsi des données FBref et des données Understat. De plus, nous centralisons également des données Transfermarkt (les données financières ainsi que notre variable cible : la valeur marchande) et du mapping issu de worldfootballR.

Nous réalisons ensuite une fusion à plusieurs niveaux : nous utilisons les identifiants connus des joueurs puis du fuzzy-mapping.

Nous devrions obtenir finalement une table prête pour de plus profondes analyses voire pour de la modélisation, mêlant ainsi des données issues des performances sportives à des données analysant la valeur marchande des joueurs de football.

## Partie A : Import de fonctions utiles à la création de la base d'apprentissage

In [4]:
# Importation des packages nécessaires

import pandas as pd
import os
import sys

# On connecte le notebook à tous les fichiers inclus dans le dossier /fonctions
sys.path.append(os.path.abspath("../fonctions"))

%load_ext autoreload
%autoreload 2

from merging import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Partie B : Chargement et préparation des sources


### B1 : Chargement et préparation des données

Nous importons dans un premier temps nos trois fichiers comprenant nos données et réalisons les modifications nécessaires à leur fusion.

Par exemple, nous regroupons toutes les blessures par saison pour chaque joueur.

In [6]:
# Chargement des données du mapping
df_mapping_initial = pd.read_csv("../data_finale/mapping_worldfootballR/mapping_fbref_tm.csv", encoding='latin1')

# Chargement des données du dataset Soccerdata
df_soccerdata_initial = pd.read_csv("../data/soccerdata/data_final_soccerdata.csv")

# Chargement des données du dataset Transfermarkt
df_players = pd.read_csv("../data/transfermarkt_datasets/players.csv")
df_valuations = pd.read_csv("../data/transfermarkt_datasets/player_valuations.csv")

# Chargement des données du dataset de blessures Transfermarkt
df_blessures = pd.read_csv("../data/dataset_blessures.csv")

# Préparation des données de Transfermarkt
df_tm_initial = prepare_transfermarkt_data(
    df_players,
    df_valuations
)

# Préparation des données de blessures Transfermarkt
df_blessures_initial = aggregate_injuries_by_season(df_blessures)

KeyError: 'player'

### B2 : Nettoyage et harmonisation des données

Plutôt que de traiter chaque DataFrame manuellement, nous utilisons la fonction centralisée `match_player_data()`. 

Cette approche permet de :
* **Standardiser les clés de jointure** : Nettoyage des caractères spéciaux, minuscules et suppression des accents sur l'ensemble des noms de joueurs.
* **Harmoniser les formats temporels** : Extraction standardisée des années de naissance et conversion des formats de saison sur un format numérique à 4 chiffres (ex: `2023`).
* **Traiter les données financières** : Dédoublonnage dynamique des valorisations Transfermarkt pour ne conserver que la dernière estimation connue par joueur et par saison.
* **Conserver l'intégrité des tables** : Traitement simultané des 4 sources de données (**FBref/Mapping**, **Soccerdata**, **Transfermarkt Valorisations** et **Transfermarkt Blessures**).

In [ ]:
# Nous appliquons les logiques décrites ci-dessus

df_mapping, df_soccerdata, df_tm, df_blessures = match_player_data(df_mapping_initial, df_soccerdata_initial,
                                                      df_tm_initial, df_blessures_initial)

## Partie C : La fusion à multi-niveaux

Nous appliquons ensuite une stratégie de fusion des bases de données en 4 étapes pour maximiser le taux de correspondance. Pour maximiser le taux de correspondance tout en garantissant la fiabilité des associations, nous appliquons une stratégie d'appariement en cascade. 

Dans un premier temps, nous réalisons une jointure exacte via le dictionnaire de mapping *worldfootballR*, via le nom et la saison. Un pipeline de correspondance par *fuzzy matching* est ensuite appliqué.

**Étape 1 — Match exact sur le Mapping (Saison) :**
   Jointure stricte entre `SoccerData` et `Transfermarkt` via le `Mapping`.

**Étape 2 — Match flou (Fuzzy) sur le Mapping (Saison) :**
   Pour les joueurs non trouvés, comparaison textuelle floue avec les entrées du mapping de la même saison (seuil de score $\ge 85-90\%$).

**Étape 3.1 — Match direct exact Transfermarkt :**
   En cas d'absence dans la table de mapping, tentative de jointure exacte directe avec la base Transfermarkt.

**Étape 3.2 — Match direct flou Transfermarkt + Validation par le Club :**
   Recherche floue directe sur les noms Transfermarkt combinée à un **score d'homologation du club**. Pour valider le match d'un nom à score intermédiaire ($75\% \le \text{score} < 95\%$), la concordance de l'équipe est vérifiée via un dictionnaire de synonymes de clubs (ex: *PSG* $\leftrightarrow$ *Paris Saint-Germain*, *M'gladbach* $\leftrightarrow$ *Borussia Mönchengladbach*).



Une fois l'appariement Transfermarkt-Soccerdata effectué, le jeu de données final est enrichi : nous fusionnons les données de blessures ainsi que les résultats collectifs à la base d'apprentissage, qui est alors complète.

In [ ]:
df_final, still_missing = run_player_matching(df_soccerdata, df_mapping, df_tm, df_blessures)

Match exact Mapping saison : 16280 | Restants : 833
Match fuzzy Mapping saison : 139 | Restants : 694
Match direct TM exact : 303 | Restants : 391
Match fuzzy TM : 97 | Restants : 293


Nous pouvons enfin importer notre base d'apprentissage sous le format CSV.

In [3]:
import pandas as pd

def taux_erreur_fuzzy(df_final, colonne_type_match='match_type', 
                        colonne_naissance_soccerdata='birth_year_soccerdata',
                        colonne_naissance_tm='birth_year_tm'):
    """
    Calcule un taux d'erreur estimé du fuzzy matching en comparant
    une variable indépendante (année de naissance) entre les deux sources
    pour les lignes matchées en mode "fuzzy".
    """
    # On isole uniquement les lignes issues d'un match flou
    df_fuzzy = df_final[df_final[colonne_type_match].str.contains('fuzzy', case=False, na=False)].copy()

    if df_fuzzy.empty:
        print("Aucune ligne fuzzy détectée — vérifiez le nom de la colonne de type de match.")
        return None

    # Comparaison de l'année de naissance (variable NON utilisée dans le matching)
    df_fuzzy['naissance_coherente'] = (
        df_fuzzy[colonne_naissance_soccerdata] == df_fuzzy[colonne_naissance_tm]
    )

    nb_incoherents = (~df_fuzzy['naissance_coherente']).sum()
    nb_total = len(df_fuzzy)
    taux_erreur = nb_incoherents / nb_total

    print(f"Nombre de matchs fuzzy analysés : {nb_total}")
    print(f"Nombre de matchs incohérents (année naissance différente) : {nb_incoherents}")
    print(f"Taux d'erreur estimé du fuzzy matching : {taux_erreur:.2%}")

    return df_fuzzy[~df_fuzzy['naissance_coherente']]  # renvoie les cas suspects pour inspection

df_erreurs_suspectes = taux_erreur_fuzzy(df_final)

NameError: name 'df_final' is not defined

In [ ]:
df_final.to_csv(r'..\data_finale\base_apprentissage.csv', index=False, sep=',', encoding='utf-8-sig')
still_missing.to_csv(r'..\data_finale\analyse_orphelins\still_missing.csv', index=False, sep=',', encoding='utf-8-sig')
df_soccerdata.to_csv(r'..\data_finale\analyse_orphelins\soccerdata.csv', index=False, sep=',', encoding='utf-8-sig')

## Partie D : Les joueurs orphelins

Certains joueurs n'ont pas réussi à être matchés et ne sont donc pas présents dans notre base d'apprentissage finale. Nous appelons ces joueurs des joueurs *"orphelins"*.

Ils sont 275 ce qui représente 4,44% de notre dataframe initial. On perçoit également que la plupart sont de jeunes joueurs ayant joué en 2025/2026.

In [ ]:
# Chargement des fichiers récemment édités

still_missing = pd.read_csv(r'..\data_finale\analyse_orphelins\still_missing.csv', encoding='utf-8-sig')
df_soccerdata = pd.read_csv(r'..\data_finale\analyse_orphelins\soccerdata.csv', encoding='utf-8-sig')

In [ ]:
# Calcul du nombre de joueurs orphelins et du taux par rapport au nombre total de joueurs

nb_joueurs_orphelins = still_missing['join_key'].nunique()
nb_joueurs_total = df_soccerdata['join_key'].nunique()

print(f"Joueurs orphelins : {nb_joueurs_orphelins}")
print(f"Taux joueurs orphelins : {nb_joueurs_orphelins / nb_joueurs_total:.2%}")

Joueurs orphelins : 275
Taux joueurs orphelins : 4.44%


In [ ]:
orphelins_par_saison = (
    still_missing
    .groupby('season_year')
    .size()
    .sort_index()
)

print(orphelins_par_saison)

season_year
2020      5
2021     19
2022     31
2023     47
2024     14
2025    177
dtype: int64


Si l'on cherche à trouver pourquoi ces joueurs sont orphelins, on perçoit que pour la plupart, c'est parce qu'ils ont changé de club au mercato d'hiver et ne sont donc plus comptabilisés dans notre périmètre du Big 5 (Angleterre, Espagne, Allemagne, Italie, France).

In [ ]:
executer_diagnostic_et_repartition(still_missing, df_tm, clean_text_func=clean_text)

,Raison de rejet (Simplifiée),Pourcentage (%)
0,Score nom OK mais club trop faible,93.52
1,Score nom + club OK — à vérifier manuellement,6.48
